![Tutorial Header](https://dagshub.com/DagsHub/hello-world-llm/raw/main/s3:/hello-world-llm/resources/llm-tutorial-header.png)

# LLM Workflow: A Step-by-Step Tutorial with DagsHub

Large Language Models (LLMs) enable building powerful applications. From RAG-based chatbots, to image editing, and even end-to-end application building.

There are many tutorials teaching parts of the workflow of LLM application creation - e.g. prompt engineering, or a RAG application, but I couldn't find examples showcasing a simple end-to-end LLM development workflow. That's what this tutorial is for.

In this tutorial, we'll go through the key stages of an LLM project-from defining the problem to evaluating the model. We'll focus on what the ML/AI team typically handles.

### What You'll Learn:

1. **Problem Definition:** How to clearly define the goal of your LLM-powered app.

1. **Data Preparation:** Using DagsHub Docs as our knowledge base, we'll chunk and embed documents and store them in a Qdrant vector DB.

1. **Model & Prompt Selection:** We'll experiment with both proprietary (OpenAI GPT-5-nano) and open-source (Gemma3-4B) models.

1. **Evaluation:** We'll test and evaluate the model using RAGAS and LLM-as-a-judge methods, logging everything to MLflow.

### Prerequisites:

* A DagsHub instance on your OpenShift cluster.
* A DagsHub account in your DagsHub instanse on your OpenShift cluster.
* Basic understanding of Python and machine learning workflows.
* (For using OpenAI GPT) an OpenAI API key.

## 👋 Introduction

Welcome to this LLM Hello World Tutorial. We will be building an end-to-end Q&A chatbot for DagsHub’s documentation with an advanced RAG pipeline, incorporate fine-tuning, and even build an evaluation flow. In this tutorial we'll take an approach that is used in real-world production LLM applications.

By the end of this tutorial, you'll have implemented a chatbot that can provide DagsHub support based on user queries.

Let's get going!

## ⚙️ Stage 0: Setup & Configuration

Before diving in, we need to configure and set up a few things. Let's install all the dependecies we'll need and configure Git for Colab.

In [ ]:
import os
os.environ["PIP_EXTRA_INDEX_URL"] = "https://pypi.org/simple/"

%pip install --upgrade pip
%pip install -q dagshub[jupyter]

# Install the relevant packages beforehand. For most of these cells, you only need the DagsHub package
%pip install -q mlflow==3.7.0
%pip install -q tiktoken==0.12.0

# RAG Requirements
%pip install -q qdrant-client==1.16.2 # VectorDB
%pip install -q sentence-transformers==5.2.0 # Tools for embedding our docs
%pip install -q "openai<3.0.0"
%pip install -q requests
%pip install -q tqdm==4.67.1
%pip install --upgrade torch==2.11.0

# Gemma Requirements
%pip install -q transformers
%pip install -q torchvision
%pip install -q accelerate
    
# Eval Requirements
%pip install -q ragas==0.4.1
%pip install -q bert-score==0.3.13


> **💡🔥 IMPORTANT:**  To be on the safe side, restart the notebook after installing the packages!

### DagsHub and Git Configuration & Auth
Let's configure the requirements to work with Git, and also get a DagsHub auth token in a variable called `TOKEN` for later authentication in Git actions.

In [ ]:
#@markdown **You need to sign up for [DagsHub](https://dagshub.com/user/sign_up) , then enter the name of the repository you'd like to create, and your username and email.**

#@markdown Enter the repository name for the project (don't worry if it doesn't exist yet, we'll create it in the next step):
REPO_NAME= "hello-world-llm" #@param {type:"string"}

#@markdown Enter the username of your DagsHub account:
USER_NAME = "INSERT YOUR DAGSHUB USERNAME HERE" #@param {type:"string"}

# Change if you want to use an organization repo
REPO_OWNER = USER_NAME

#@markdown Enter the email for your DagsHub account:
EMAIL = "INSET THE EMALI HERE" #@param {type:"string"}

import os
HOST = os.getenv("DAGSHUB_CLIENT_HOST")
#@markdown ---

import IPython

!git config --global user.email {EMAIL}
!git config --global user.name {USER_NAME}

In [ ]:
# Connecting to the OpenShift Deployed DagsHub
import dagshub
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, host=HOST)

TOKEN = dagshub.auth.get_token()

> 💡**Note 2: DagsHub Token Environment Variable** For automation purposes, you can set your DagsHub Token as `DAGSHUB_USER_TOKEN` environment variables*

Before starting, we need to add the dataset to our DagsHub repository.

In [ ]:
dagshub.upload_files(repo=f"{REPO_OWNER}/{REPO_NAME}", local_path="docs", bucket=True)

In [ ]:
from dagshub.data_engine import datasources


# Connect to the RAG dataset
doc_dataset = datasources.get_or_create(repo=f"{REPO_OWNER}/{REPO_NAME}",
                                           name="docs",
                                           path=f"s3://{REPO_NAME}/docs")

In [ ]:
docs_file_path = [dp["path"] for dp in doc_dataset.select("path").all()]
docs_file_path

---

## 🧹 Stage 1: ML Problem Definition

**What:**  
Define the real-world LLM task clearly in plain text - no code required. Our goal is to create a DagsHub documentation support chatbot. It should answer user queries based on official docs, providing helpful, accurate, and concise responses. The core of the system is a RAG pipeline, combining document retrieval with LLM generation.

**Why:**  
A clearly defined problem ensures alignment across the project. For this chatbot, success means:

* Relevant answers pulled from the latest docs.

* High-quality, user-friendly language.

* Reduced manual support overhead.

**Real-World Impact:**  
Such a chatbot improves customer experience, reduces response time, and eases the load on human support teams. It also showcases how to integrate and fine-tune LLMs effectively in production.

> 🛎️ **Tip:** Keep in mind who your users are. Our chatbot targets developers familiar with DagsHub, so accuracy and relevance are crucial.

---

## 📊 Stage 2: Data Preparation

**Retrieval-Augmented Generation (RAG)** involves augmenting an LLM with external knowledge. The first step is to prepare a dataset that the LLM can retrieve information from during generation. This could be a collection of documents, knowledge base articles, Q&A pairs, etc., relevant to your use case.

**What:**  
We'll process DagsHub's documentation stored in a Git repository. Key steps include:

1. Loading all doc files.

1. Cleaning and chunking the content into manageable pieces (~1000 tokens each).

1. Embedding each chunk using a transformer model.

1. Storing vectors in Qdrant for efficient retrieval.

**Why:**  
Chunking the docs helps LLMs retrieve only relevant content due to context window limits. Metadata (like section names and source paths) will help track and organize content effectively.

<br/>

> 💡 **Note:** Real-world datasets often involve PDFs or scanned documents requiring OCR. For simplicity, we’re using Markdown docs, but your project might require adopting tools like [Tesseract OCR](https://github.com/tesseract-ocr/tesseract) or [EasyOCR](https://github.com/JaidedAI/EasyOCR) similar methods apply.

### 2.1 Load and chunk the documents
Since LLMs have context length limits, we'll split each doc into smaller, logically coherent chunks (~1000 tokens) with overlaps to preserve context.

> 🛎️ **Tip:** Pay attention to how we chunk the data. Poor chunking can cut off important information, affecting retrieval quality later.
>
> For example, originally when working on this tutorial, I used 500 characters per chunk, with a 50 character overlap. However, while testing sample queries I noticed that many chunks are cut in the middle and don't provide enough information to generate an answer.
>
> Other considerations include whether to embed metadata alongside the content.

In [ ]:
import re
import pandas as pd
from pathlib import Path

document_rows = []

# Clean the text in the 'description' column
def clean_text(text):
  # Remove unwanted block tags and their content
  text = re.sub(r'<(style|script|iframe).*?>.*?</\1>', '', text, flags=re.DOTALL|re.IGNORECASE)

  # Remove any remaining HTML tags
  text = re.sub(r'<.*?>', '', text)
  return text.lower().strip()

# Function to chunk text into segments of approximately 500 words, with 50
# character overlap to maintain context between chunks
def chunk_text(text, chunk_size=1000, overlap=300):
  chunk_list = []
  # Split by logical paragraphs
  paragraphs = re.split(r"\n\s*\n", text)
  chunk = ""
  for para in paragraphs:
    if len(chunk) + len(para) < chunk_size:
      chunk += para + "\n\n"
    else:
      chunk_list.append(chunk.strip())
      # start new chunk, include overlap from end of previous chunk
      chunk = chunk[-overlap:] + para + "\n\n"
  if chunk:
    chunk_list.append(chunk.strip())
  return chunk_list

Now let's run the cleaning and chunking on our documents.

In [ ]:
for dp in doc_dataset.all():
  file_path = dp.download_file()
  text = file_path.read_text()
  cleaned_text = clean_text(text)
  chunks = chunk_text(cleaned_text)

  orig_path = Path(dp["path"])
  section = orig_path.parts[0] if len(orig_path.parts) > 1 else "general"

  for chunk in chunks:
    document_rows.append({
      "chunk": chunk,
      "source_path": dp["path"],
      "source_file_url": dp["dagshub_download_url"],
      "section": section
    })

documents_df = pd.DataFrame(document_rows)

print(f"Total chunks: {len(documents_df)}")
print(f"Sample chunk: \n{documents_df.loc[0]['chunk']}")

This code ensures we break on paragraph boundaries when possible and keep a slight overlap between chunks to avoid losing context. Each chunk will be an independent piece of content we can embed and search.

### 2.3. Embed the chunks
We’ll convert each text chunk into a vector embedding for semantic search.

To use a more up to date approach, we'll combine "regular" dense embedding with a sparse embedding option for reranking and getting better results. To do this efficiently, we'll use the FastEmbed package from Qdrant alongside Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model_string = "Qwen/Qwen3-Embedding-0.6B"
embedder = SentenceTransformer(model_string)

This step will work much faster on a GPU machine - For some reason running this in a regular notebook doesn't show progress.

In [ ]:
embeddings = embedder.encode(list(documents_df["chunk"]), prompt="passage: ", batch_size=32, show_progress_bar=True)

Let's add it to our previously created dataframe:

In [ ]:
import json
documents_df["embeddings"] = [json.dumps(x) for x in embeddings.tolist()]
documents_df

### 2.4. Save the RAG dataset back to DagsHub
Since we don't want to reprocess the data every time, let's save the current state to DagsHub as a Data Engine dataset before spinning up our vectorDB.

In [ ]:
import os
import uuid

# Create folder if it doesn't exist
os.makedirs('rag_data', exist_ok=True)

def convert_to_files(chunk):
  # Making the filename content dependent
  filename = f'chunk_{uuid.uuid5(namespace=uuid.NAMESPACE_DNS, name=chunk).hex}.txt'
  filepath = os.path.join('rag_data', filename)

  # Write chunk to file
  with open(filepath, 'w', encoding='utf-8') as f:
    f.write(chunk)

  # Store relative path
  return filename

# Add the path column
documents_df['path'] = documents_df['chunk'].apply(convert_to_files)

In [ ]:
# Upload chunk files to DagsHub
dagshub.upload_files(repo=f"{REPO_OWNER}/{REPO_NAME}", local_path="rag_data", bucket=True)

In [ ]:
from dagshub.data_engine import datasources

# Create Datasource for chunks and add metadata
rag_datasource = datasources.get_or_create(repo=f"{REPO_OWNER}/{REPO_NAME}",
                                           name="rag_data",
                                           path=f"s3://{REPO_NAME}/rag_data")

We've created our datasource, but it's just a bunch of indexed files for now. Let's add all of the metadata + embeddings we created so that our datasource state is fully versioned:

In [ ]:
rag_datasource.upload_metadata_from_dataframe(documents_df.drop(columns=["chunk"]), path_column="path")

print("Check out the newly created datasource here:")
rag_datasource.visualize()

### 2.5. Initialize vector database
For this tutorial, we'll use Qdrant. Qdrant is a powerful vector DB and we can run it directly in Colab for convenience.

> 🛎️ **Pro Tip:** You can extend this to include metadata fields (like section names) to filter or rank results more intelligently.

Let's create an local instance and upload our embeddings:

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import json

# Use in-memory Qdrant for convenience
qdrant_client = QdrantClient(":memory:")

collection_name = "dagshub_docs"

rag_data = rag_datasource.all()
rag_data.download_files(".")
vector_df = rag_data.dataframe

embedding_dim = len(json.loads(vector_df.loc[0]["embeddings"]))

qdrant_client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.DOT),
)

Let's insert all our document chunks:

In [ ]:
points = vector_df.reset_index(drop=True).apply(lambda row: PointStruct(
    id=row.name,
    vector=json.loads(row["embeddings"]),
    payload={
        "text": open(f'rag_data/{row["path"]}').read(),
        "chunk_path": row["path"]
    }), axis=1).tolist()
qdrant_client.upsert(collection_name=collection_name, points=points)

That's it. Our data is indexed and ready for vector search!

Before moving on to the next segment, let's test it out:

In [ ]:
question = "How can I log an MLflow experiment on DagsHub?"

search_res = qdrant_client.query_points(
  collection_name=collection_name,
  query=embedder.encode(question).tolist(),
  limit=3,  # Return top 3 results
).points

retrieved_lines_with_distances = [
  (hit.payload["text"], hit.score) for hit in search_res
]

context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

print(json.dumps(retrieved_lines_with_distances, indent=4))

As you can see from a quick review, it does seem like the chunks retrieved are related to the question, and indeed the first chunk (Highest score) contains the answer!

🙌 Take a moment to appreciate the progress so far. You just built a vector DB with some real data. You can easily swap this out for your own dataset.

With our retrieval system done, it's time to engineer our prompts and protype our LLM application!

---

## 🤖 Stage 3: Model & Prompt Selection
Model and prompt selection are the focus of many LLM tutorials online. Though they are well-covered, we'll see how they fit into a typical LLM development flow below.

Our goal is to effectively build a matrix of models and prompts, to find the optimal combo for our specific needs.

**What:**  
We'll test both:

* OpenAI's GPT-5-nano for high-quality, but cheap and fast, outputs.

* Lightweight open-source models (like Gemma4-E2B-it) that can be fine-tuned locally.

Prompt engineering will be handled via system prompts, carefully crafted to produce clear and concise responses. We'll also log different prompts, models, and hyperparameters (temperature, max tokens, etc.) to track performance.

**Why:**  
[Effective prompt engineering](https://applied-llms.org/#focus-on-getting-the-most-out-of-fundamental-prompting-techniques) can significantly enhance your application's performance.

By testing multiple prompts with each model, we can understand how well the model retrieves and generates responses in the restaurant domain.

> 💡 **Reminder:** We'll log each model, prompt, and hyperparameter combo (temperature, max_tokens, etc.) to MLflow for easy comparison.

### 3.1. LLM input structure (and a test)
Below we've broken down the components of the input into the model. Note the breakdown into the `system_prompt` and `user_prompt`:
* The system prompt is our instruction to the LLM on HOW to perform it's task.
* The user prompt is a way to structure the input so that the LLM understands it. In our case this combines the retrieved context, with the user question.

> 💡 **Note:** See how the user prompt starts with a paragraph explaining the structure to the LLM. This isn't a system prompt since it isn't instructing the LLM what to do, but rather what it's going to get next.

In [ ]:
import mlflow

system_prompt = """
You are a helpful assistant on all matters related to DagsHub, the AI platform.
Respond to the user's message as a friendly AI.
"""

mlflow.genai.register_prompt(
            name=f"support-prompt_base",
            template=system_prompt,
            commit_message="Initial commit"
        )

user_prompt = """
Use the following pieces of information enclosed in <context> tags to provide
an answer to the question enclosed in <question> tags. If you don't know the answer,
just say that you don't know.

<context>
{context}
</context>

<question>
{question}
</question>
"""

messages = [
  {"role": "system", "content": system_prompt},
  {"role": "user", "content": user_prompt.format(context=context, question=question)}
]

The last `messages` object is the format in which we'll feed our inputs to the LLMs. Let's see an example with GPT-5-nano for the inputs and outputs:

In [ ]:
import openai
from getpass import getpass

openai.api_key = getpass("Paste your OpenAI API Key:")

# Load OAI Client
oai_client = openai.OpenAI(api_key=openai.api_key)

In [ ]:
response = oai_client.chat.completions.create(
    model = "gpt-5-nano",
    messages = messages,
    max_completion_tokens = 512,
    response_format={"type": "text"},
    reasoning_effort="minimal",
)

print(response.choices[0].message.content)

### 3.2. Standardizing the prompt engineering flow
In the real world we want to create functions to provide the context for different questions, support different prompts, and even different models. In other words, we want to make it easy to experiment.

To make experimentation easy, we need to break up our prompting into easily editable components:
1. We need a class for the model outputs since we'll be using a new OSS model, we want the format it returns to be similar to OpenAI's output - we'll call this the `ChatCompletion` class.
1. We need a function to retrieve the context from the vectorDB we created earlier - we'll call this the `retrieve_context` function.
1. We need a function to compine the context and the user question and create the `messages` object - we'll call it the `craft_prompt` function.
1. Since we plan to compare models from different providers, we need to abstract the request such that we can provide different model implementations and get the same input and output interface - this will be called the `ask_assistant` function, and it'll help us accelerate experimentation.

> 💡 **Note:** that all the functions above are decorated with `@mlflow.trace()`. This will enable us to track the various stage of prompting, context building, and response generation to get better granularity in our experiments.

Let's see how we can build that:

In [ ]:
from pydantic import BaseModel
from typing import List
import mlflow
from mlflow.entities import SpanType

# An OpenAI-like ChatCompletion Object
class Message(BaseModel):
    content: str
    role: str = "assistant"

class Choice(BaseModel):
    message: Message

class ChatCompletion(BaseModel):
    choices: List[Choice]
    object: str = "chat.completion"

# Functions for tracing
@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve_context(question):
    search_res = qdrant_client.query_points(
      collection_name=collection_name,
      query=embedder.encode(question).tolist(),
      limit=3,  # Return top 3 results
    ).points

    retrieved_lines_with_distances = [
      (hit.payload["text"], hit.score) for hit in search_res
    ]

    context = [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]

    if mlflow.active_run() and rag_datasource:
        q = rag_datasource
        for hit in search_res:
          q = q | (rag_datasource["path"] == hit.payload["chunk_path"])
        # This will run the query and create the reference on DagsHub
        q.all()

    return context

@mlflow.trace(span_type=SpanType.PARSER)
def craft_prompt(context, messages):
  return user_prompt.format(context=context, question=question)

  return messages

@mlflow.trace(span_type=SpanType.AGENT)
def ask_assistant(messages, system_prompt, user_prompt, llm):
  question = messages[0]['content']


  context = retrieve_context(question)
  context_str = "\n".join(context)

  messages.extend([
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": craft_prompt(context_str, question)}
  ])

  completion = llm(messages)

  return completion

The last step is implementing the specific model functions for GPT-5-nano and Gemma4-E2B-it:

In [ ]:
openai_model_name = "gpt-5-nano"
oss_model_name = "google/gemma-4-E2B-it"

In [ ]:
# Load OAI Model
oai_client = openai.OpenAI(api_key=openai.api_key)

openai_params = {
    'max_tokens': 512
    }

# OAI Model Inference
@mlflow.trace(name=openai_model_name, span_type=SpanType.CHAT_MODEL)
def gpt(messages):
  response = oai_client.chat.completions.create(
      model = openai_model_name,
      messages = messages,
      max_completion_tokens = openai_params['max_tokens'],
      response_format={"type": "text"},
      reasoning_effort="minimal",
  )

  answer = response.choices[0].message.content
  print(answer)

  return response

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

# Load OSS Model
oss_params = {
    'temperature': 1.0,
    'top_p': 0.95,
    'top_k': 64,
    'max_tokens': 512,
    }

# Load model
processor = AutoProcessor.from_pretrained(oss_model_name)
model = AutoModelForCausalLM.from_pretrained(
    oss_model_name,
    dtype="auto",
    device_map="auto",
)

In [ ]:
# OSS Model Inference
@mlflow.trace(name=oss_model_name, span_type=SpanType.CHAT_MODEL)
def gemma(messages):
  text = processor.apply_chat_template(
      messages,
      add_generation_prompt = True, # Must add for generation,
  )

  inputs = processor(text=[text], return_tensors = "pt").to(model.device)
  with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens = oss_params['max_tokens'], # Increase for longer outputs!
        temperature = oss_params['temperature'], 
        top_p = oss_params['top_p'], 
        top_k = oss_params['top_k'],
        use_cache = True,
    )

  answer = processor.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
  print(answer)

  response = ChatCompletion(
      choices = [Choice(message = Message(content = answer))]
  )

  return response

### 3.3. Prompt engineering dry run
Now let's create and log our first prompt to test question from above with both models:

In [ ]:
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME)

# We get a link to view our experiment on our MLflow server
mlflow.tracing.disable_notebook_display()

In [ ]:
run_name = f"{openai_model_name}_base"

# To start logging to MLflow
with mlflow.start_run(run_name=run_name):
  messages = [{'role': 'user', 'content': question}]

  # OAI Model Answer
  completion = ask_assistant(messages, system_prompt, user_prompt, gpt)

  mlflow.openai.log_model(
      model=openai_model_name,
      task=oai_client.chat.completions,
      messages=messages,
      name="model",
  )

The model is fast, but note that it didn't really answer concisely.

Now let's see the OSS model (the first inference will be slower than future ones, don't worry):

In [ ]:
from transformers import AutoTokenizer

run_name = f"{oss_model_name}_base"

tokenizer = AutoTokenizer.from_pretrained(oss_model_name, padding_side="left")

with mlflow.start_run(run_name=run_name):
  messages = [{'role': 'user', 'content': question}]

  # OSS Model Answer
  completion = ask_assistant(messages, system_prompt, user_prompt, gemma)

  mlflow.transformers.log_model(
      transformers_model={
          "model": model,
          "tokenizer": tokenizer,
      },
      processor=processor,
      task="text-generation",
      name="model",
      save_pretrained=False,
      pip_requirements=[
        "mlflow==3.5.1",
        "transformers==5.5.1",
        "torch==2.11.0",
        "torchvision==0.26.0",
        "accelerate==1.13.0",
    ],
  )

Pretty great response if you ask me.

Along the way we've logged these prompts and responses to our DagsHub repo for tracking and future analysis. You can go to your experiments tab to see the results including the traces (to see traces click on the run ID to go to the integrated MLflow UI and select the "traces" tab).

### 3.4. Full prompt engineering
Now that we've tested everything out, we can build our matrix. To do this, let's stick with our single question (in reality we will probably have more than one).

We'll iterate over 2 dimensions - the model, and the prompt. In the model dimension we have OpenAI and our OSS Gemma3 model, and for prompts, we'll create 3 additional prompts (in addition to the original above):

1. In our case, you can see that the answers I'm getting are long so let's modify the system prompt to ask for a more concise response.

1. Also, let's add another prompt to make sure the model chooses the first answer if there is more than one.

1. Finally, let's add a third option for fun.

For the purpose of scoring, I'm also writing what I think is the "correct" or reference answer to the question. We'll use this to calculate the similarity between the generated answer and the correct answer, so that we can quantify which models are better.

In [ ]:
question = "How can I log an MLflow experiment on DagsHub?"

reference_answer = "Simply run `pip install dagshub mlflow`, then `dagshub.init(repo_owner='', repo_name='')`."

# Define candidate prompts for restaurant recommendations
prompt_candidates = [
    """
    You are a helpful assistant on all matters related to DagsHub, the AI platform.
    Respond to the user's message as a friendly AI.
    """,
    """
    You are a helpful assistant on all matters related to DagsHub, the AI platform.
    Respond to the user's message as a friendly but concise AI.
    """,
    """
    You are a helpful assistant on all matters related to DagsHub, the AI platform.
    Respond to the user's message as a friendly but concise AI. If there are
    multiple answers, provide the simplest one.
    """,
    """
    You are an assistant that speaks like a pirate, on all matters related to
    DagsHub, the AI platform. Respond to the user's message as a friendly but
    concise AI.
    """
]

model_candidates = {
    "openai": gpt,
    "gemma": gemma,
}

Now let's iterate over the various prompts and models, and log the traces, parameters and metrics to MLflow. We'll generate a score for each run, which will enable us to determine which prompt and model combo are the best.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

exp = mlflow.set_experiment("prompt_engineering")

# Iterate through each model and prompt combination
for model_name, model_func in model_candidates.items():
  for idx, system_prompt in enumerate(prompt_candidates):
    with mlflow.start_run(run_name=f"{model_name}_prompt_{idx+1}"):
        messages = [{'role': 'user', 'content': question}]

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("prompt", system_prompt)
        mlflow.genai.register_prompt(
            name=f"support-prompt_{idx}",
            template=system_prompt,
            commit_message="Initial commit"
        )

        completion = ask_assistant(messages, system_prompt, user_prompt, model_func)
        answer = completion.choices[0].message.content

        mlflow.log_text(answer, artifact_file="response.txt")

        # Calculate a metric for the prompts
        # Get embeddings for both answers
        gen_embedding = embedder.encode([answer])
        ref_embedding = embedder.encode([reference_answer])

        # Compute cosine similarity
        similarity = cosine_similarity(gen_embedding, ref_embedding)[0][0]
        mlflow.log_metric("similarity", similarity)

        mlflow.log_table(data={
            "prompt": [system_prompt],
            "model": [model_name],
            "response": [answer],
            "similarity score": [similarity],
            "run_id": [mlflow.active_run().info.run_id]
        }, artifact_file="prompt_eval_results.json")
        print(f"\nModel: {model_name}\nPrompt: {system_prompt}\nSimilarity to reference answer: {similarity}\n------\n")

After the run is complete, you'll be able to see a new experiment in your DagsHub MLflow UI. Since we logged our table, we'll also be able to compare the different model response (so that in addition to the similarity score, we can make sure that the best response makes sense).

Seems like Gemma with the original prompt yields the best result!

![Prompt Engineering](https://dagshub.com/DagsHub/hello-world-llm/raw/main/s3:/hello-world-llm/resources/prompt-engineering.png)

---

## ✅ Stage 4: Testing & Evaluation

**What:**  
We'll evaluate model performance using:

* Embedding-based similarity metrics.

* LLM-as-a-judge evaluations (using GPT-5.2).

* RAGAS framework to assess relevance, faithfulness, and toxicity.

**Why:**  
Evaluation is like unit testing for LLM apps—it ensures reliability before deploying to production.

All results, metrics, and runs will be logged to MLflow and visualized via DagsHub for transparency and reproducibility.

### 4.1. Evaluation Preparation

We need to set up everything we need to run our evaluation.

Starting with a dataset `test_data` - Each entry in pairs a real-world user question with its reference answer. This is one way we'll be able to measure model performance by matching the generations with the "gold standard" responses.

In [ ]:
test_data = [
    {
        "question": "What is DagsHub?",
        "reference_answer": ("DagsHub is a collaborative platform for data science "
                              "that integrates Git, DVC, and MLflow to manage code, data, and experiments.")
    },
    {
        "question": "How do I connect a dataset to DagsHub?",
        "reference_answer": ("You can connect a dataset to DagsHub using the DagsHub Data Engine "
                              "or by linking your data via DVC in your repository.")
    },
    {
        "question": "What is MLflow and how is it used in DagsHub?",
        "reference_answer": ("MLflow is an open-source platform for managing the machine learning lifecycle. "
                              "DagsHub hosts an MLflow tracking server to log experiments and models.")
    },
    {
        "question": "How does the DagsHub Data Engine help manage data?",
        "reference_answer": ("The DagsHub Data Engine enables you to version, track, and query datasets directly within your DagsHub repository.")
    }
]

Let's break this into separate datapoints and create a DagsHub datasource from this, so that we can make our evals reproducible too:

In [ ]:
# Create output directory
output_dir = "eval_data"
os.makedirs(output_dir, exist_ok=True)

for idx, entry in enumerate(test_data):
    # Write to separate JSON file
    file_path = os.path.join(output_dir, f"eval_{idx+1}.json")
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(entry, f, ensure_ascii=False, indent=2)

In [ ]:
# Upload chunk files to DagsHub
dagshub.upload_files(repo=f"{REPO_OWNER}/{REPO_NAME}", local_path="eval_data", bucket=True)

# Create Datasource for chunks and add metadata
eval_datasource = datasources.get_or_create(repo=f"{REPO_OWNER}/{REPO_NAME}",
                                           name="eval_data",
                                           path=f"s3://{REPO_NAME}/eval_data")

eval_data = []
for dp in eval_datasource.all():
  path = dp.download_file()
  eval_data.append(json.load(open(path, 'r', encoding='utf-8')))

Now that we have our test data in place, it’s time to define how we’ll evaluate the model’s responses. Below are implementations of 3 complementary evaluation methods. Later in the tutorial we'll also use RAGAS.

1.	`compute_cosine_similarity`: Measures how semantically similar the generated response is to the reference answer, based on embeddings.
1.	`compute_bertscore`: Another semantic similarity metric that considers token-level alignment using a pre-trained language model.
1.	`llm_judge`: We leverage GPT-5.2 itself to critically assess each response. It rates correctness, relevance, and completeness on a scale of 1 to 10 and provides a brief explanation.

Each of these techniques gives us a different perspective on the chatbot’s quality. Next, we’ll combine them into a unified evaluation function and apply it to our model’s outputs.

In [ ]:
from ragas import evaluate, metrics
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score

def compute_cosine_similarity(text1, text2):
    """Compute cosine similarity between two texts using the embedder."""
    emb1 = embedder.encode([text1])
    emb2 = embedder.encode([text2])
    return cosine_similarity(emb1, emb2)[0][0]

def compute_bertscore(text1, text2):
    """Compute BERTScore F1 between two texts."""
    P, R, F1 = score([text1], [text2], lang="en", verbose=False)
    return F1.item()

def llm_judge(question, context, reference, generated):
    """
    Call GPT-5.2 (via OpenAI API) to evaluate the generated answer.
    The prompt instructs GPT-5.2 to rate the answer from 1 to 10 on correctness, relevance, and completeness,
    and provide a brief explanation.
    """
    oai_client = openai.OpenAI(api_key=openai.api_key)

    # Craft a prompt for evaluation.
    prompt = (
        "You are an expert AI evaluator. Evaluate the following answer to the question based on correctness, "
        "relevance, and completeness. Return your answer in JSON format as: {\"score\": <number between 1 and 10>, \"explanation\": \"...\"}.\n\n"
        f"Question: {question}\n"
        f"Context: {context}\n"
        f"Reference Answer: {reference}\n"
        f"Generated Answer: {generated}\n"
        "Your evaluation:"
    )

    try:
        response = oai_client.chat.completions.create(
            model = "gpt-5.4",
            messages = [
                {"role": "system", "content": "You are a helpful evaluator."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens = 512,
            response_format = {
                  "type": "json_schema",
                  "json_schema": {
                      "name": "judge",
                      "strict": True,
                      "schema": {
                          "type": "object",
                          "properties": {
                              "score": {"type": "integer"},
                              "explanation": {"type": "string"},
                          },
                          "required": ["score", "explanation"],
                          "additionalProperties": False,
                      },
                  },
              },
        )
        reply = response.choices[0].message.content

        # Try to parse the response as JSON
        evaluation = json.loads(reply)
    except Exception as e:
        print("Error during LLM evaluation:", e)
        evaluation = {"score": None, "explanation": reply}
    return evaluation

With our evaluation methods defined, let’s pull everything together. The evaluate_answer function runs the full evaluation loop for each query: it compares the generated answer to the reference using cosine similarity and BERTScore, then gets an expert judgment from GPT-5.2.

Additionally, we’ve set up a ragas_eval function to evaluate the dataset at scale using RAGAS metrics. This helps us measure key traits like response relevancy, factual faithfulness, and how well the retrieved contexts contribute to accurate answers.

In [ ]:
def evaluate_answer(question, reference, generated, context):
    """
    Given a question, reference answer, generated answer, and retrieved context,
    compute cosine similarity, BERTScore, and get GPT-5.2 evaluation.
    Returns a dictionary with all metric values.
    """
    similarity = compute_cosine_similarity(generated, reference)
    bert_results = compute_bertscore(generated, reference)

    judge_results = llm_judge(question, "\n".join(context), reference, generated)

    results = {
        "user_input": question,
        "retrieved_contexts": context,
        "reference": reference,
        "response": generated,
        "cosine_similarity": similarity,
        "bert_score_f1": bert_results,
        "llm_judge_score": judge_results.get("score"),
        "llm_judge_explanation": judge_results.get("explanation")
    }
    return results

def ragas_eval(samples):
    from ragas import evaluate, metrics, EvaluationDataset
    from ragas.llms import llm_factory
    from ragas.embeddings import HuggingFaceEmbeddings

    # Configure LLM and embeddings for RAGAS
    evaluator_llm = llm_factory("gpt-4o-mini", client=oai_client)
    evaluator_embeddings = HuggingFaceEmbeddings(model=model_string)

    evaluation_dataset = EvaluationDataset.from_list(samples)

    # Evaluate using RAGAS metrics
    ragas_results = evaluate(
        evaluation_dataset,
        metrics=[metrics.ResponseRelevancy(), metrics.Faithfulness(), metrics.ContextPrecision()],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings
    )
    return ragas_results

Now that our evaluation functions are ready, we can run them across our full test set and track everything in MLflow.

This next block does exactly that:
* It loops through each test sample, generates an answer using the selected model, and evaluates it using all our defined metrics.
* It logs each result, both individually and in aggregate, to DagsHub via MLflow—including BERTScore, cosine similarity, and GPT-5.2 judgment.
* It also logs RAGAS metrics for a more holistic view of the chatbot’s performance.

Finally, we log the evaluated model itself so the run is fully traceable and reproducible. Up next, we’ll inspect the results and see how well the model performed.

In [ ]:
import pandas as pd
from datetime import datetime

exp = mlflow.set_experiment("llm_evaluation")

# Defining which model to evaluate
eval_model_name = openai_model_name
eval_model_func = gpt

# Evaluate each sample using the provided generated answer
evaluation_results = []

with mlflow.start_run(
    experiment_id=exp.experiment_id,
    run_name=f"eval_{eval_model_name}_{datetime.now().strftime('%Y-%m-%d-%H-%M-%s')}"
):
    eval_datasource.all()

    mlflow.transformers.autolog()
    mlflow.log_param("model_name", eval_model_name)

    for sample in eval_data:
        q = sample["question"]
        ref = sample["reference_answer"]
        messages = [{'role': 'user', 'content': q}]

        # So we don't double-trace the context retrieval
        mlflow.tracing.disable()
        context = retrieve_context(q)
        mlflow.tracing.enable()

        response = ask_assistant(
            messages,
            system_prompt,
            user_prompt,
            eval_model_func,
            )
        gen = response.choices[0].message.content
        result = evaluate_answer(q, ref, gen, context)
        evaluation_results.append(result)

    df_results = pd.DataFrame(evaluation_results)
    mlflow.log_table(data=df_results, artifact_file="evaluation_results.json")

    ragas_results = ragas_eval(evaluation_results)
    mlflow.log_table(data=ragas_results.to_pandas(), artifact_file="ragas_results.json")

    mlflow.log_metrics({
        "avg_bert_score_f1": float(df_results["bert_score_f1"].mean()),
        "avg_answer_similarity": float(df_results["cosine_similarity"].mean()),
        "avg_norm_llm_judge_score": float(df_results["llm_judge_score"].mean() / 10),
    })

    # We want to log the model here too, so that we can associate the eval with the correct model version
    mlflow.openai.log_model(
        model=openai_model_name,
        task=oai_client.chat.completions,
        messages=messages,
        name="model",
    )

That's it! We've evaluated our model successfully. Below you can see the resulting report, logged to our DagsHub MLflow server.

The model isn't amazing, but that's what evals are for. Opportunities for improvement are to add reranking for the retrieval steps as the context seems suboptimally relevant, and to improve the prompting.

![Eval Result 1](https://dagshub.com/DagsHub/hello-world-llm/raw/main/s3:/hello-world-llm/resources/eval-1.png)

On DagsHub we also have our retrieved context, as well as the eval dataset logged and connected.

![Eval Result 2](https://dagshub.com/DagsHub/hello-world-llm/raw/main/s3:/hello-world-llm/resources/eval-2.png)

## 🐶 Conclusion

Congratulations! You’ve just built and evaluated a production-style LLM support chatbot from scratch. Along the way, you:
* Defined a real-world problem and translated it into a solvable LLM task.
* Prepared and chunked documentation into a searchable format.
* Embedded and indexed data in a vector database (Qdrant).
* Designed and tested prompt variations across OpenAI and OSS models.
* Evaluated everything with both quantitative metrics and LLM-based judgment.
* Logged and tracked results, models, and prompts with MLflow in DagsHub.

This workflow is the backbone of many real-world GenAI systems and now you’ve seen how each part fits together.

🚀 Want to go further?
Try adding multilingual support, expanding the evaluation dataset, or plugging the model into a real frontend. You can even turn this into a Slack bot or VS Code assistant.

Thanks for following along. I hope this was helpful and a bit fun too :)
If you build something cool with it, tag @TheRealDagsHub on Twitter, we’d love to see it!

In [ ]:
from dagshub.notebook import save_notebook

save_notebook(repo=f"{REPO_OWNER}/{REPO_NAME}", path="hello_world_llm.ipynb")